In [1]:
import os

In [2]:
%pwd

'c:\\Users\\amrut\\Downloads\\kidney-Disease-Classifiaction-DLProject\\research'

In [5]:


os.chdir(r"C:\Users\amrut\Downloads\kidney-Disease-Classifiaction-DLProject")

print(os.getcwd())

C:\Users\amrut\Downloads\kidney-Disease-Classifiaction-DLProject


In [6]:
%pwd

'C:\\Users\\amrut\\Downloads\\kidney-Disease-Classifiaction-DLProject'

In [7]:
from dataclasses import dataclass
from pathlib import Path


@dataclass(frozen=True)
class TrainingConfig:
    root_dir: Path
    trained_model_path: Path
    updated_base_model_path: Path
    training_data: Path
    params_epochs: int
    params_batch_size: int
    params_is_augmentation: bool
    params_image_size: list

In [8]:
from cnnClassifier.constants import *
from cnnClassifier.utils.common import read_yaml, create_directories
import tensorflow as tf

In [9]:
class ConfigurationManager:
    def __init__(
        self,
        config_filepath = CONFIG_FILE_PATH,
        params_filepath = PARAMS_FILE_PATH):

        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)

        create_directories([self.config.artifacts_root])


    
    def get_training_config(self) -> TrainingConfig:
        training = self.config.training
        prepare_base_model = self.config.prepare_base_model
        params = self.params
        training_data = os.path.join(self.config.data_ingestion.unzip_dir, "kidney-ct-scan-image")
        create_directories([
            Path(training.root_dir)
        ])

        training_config = TrainingConfig(
            root_dir=Path(training.root_dir),
            trained_model_path=Path(training.trained_model_path),
            updated_base_model_path=Path(prepare_base_model.updated_base_model_path),
            training_data=Path(training_data),
            params_epochs=params.EPOCHS,
            params_batch_size=params.BATCH_SIZE,
            params_is_augmentation=params.AUGMENTATION,
            params_image_size=params.IMAGE_SIZE
        )

        return training_config

In [10]:
import os
import urllib.request as request
from zipfile import ZipFile
import tensorflow as tf
import time

In [11]:
import math
import tensorflow as tf
class Training:
    def __init__(self, config: TrainingConfig):
        self.config = config

    
    def get_base_model(self):
        self.model = tf.keras.models.load_model(
            self.config.updated_base_model_path
        )

    def train_valid_generator(self):

        datagenerator_kwargs = dict(
            preprocessing_function=tf.keras.applications.vgg16.preprocess_input,
            validation_split=0.20
        )

        dataflow_kwargs = dict(
            target_size=self.config.params_image_size[:-1],
            batch_size=self.config.params_batch_size,
            interpolation="bilinear"
        )

        valid_datagenerator = tf.keras.preprocessing.image.ImageDataGenerator(
            **datagenerator_kwargs
        )

        self.valid_generator = valid_datagenerator.flow_from_directory(
            directory=self.config.training_data,
            subset="validation",
            shuffle=False,
            **dataflow_kwargs
        )

        if self.config.params_is_augmentation:
            train_datagenerator = tf.keras.preprocessing.image.ImageDataGenerator(
                rotation_range=40,
                horizontal_flip=True,
                width_shift_range=0.2,
                height_shift_range=0.2,
                shear_range=0.2,
                zoom_range=0.2,
                **datagenerator_kwargs
            )
        else:
            train_datagenerator = valid_datagenerator

        self.train_generator = train_datagenerator.flow_from_directory(
            directory=self.config.training_data,
            subset="training",
            shuffle=True,
            **dataflow_kwargs
        )
        
    @staticmethod
    def save_model(path: Path, model: tf.keras.Model):
        model.save(path)
    
    def train(self):
        self.steps_per_epoch = math.ceil(
        self.train_generator.samples /
        self.train_generator.batch_size
    )

        self.validation_steps = math.ceil(
        self.valid_generator.samples /
        self.valid_generator.batch_size
    )

        checkpoint = tf.keras.callbacks.ModelCheckpoint(
        filepath=str(self.config.trained_model_path),
        monitor="val_loss",
        save_best_only=True,
        verbose=1
    )

        self.model.fit(
        self.train_generator,
        epochs=self.config.params_epochs,
        steps_per_epoch=self.steps_per_epoch,
        validation_data=self.valid_generator,
        validation_steps=self.validation_steps,
        callbacks=[checkpoint]
    )


In [60]:

import os

DATASET_PATH = os.path.join(
    "artifacts",
    "data_ingestion",
    "CT-KIDNEY-DATASET-Normal-Cyst-Tumor-Stone",
    "CT-KIDNEY-DATASET-Normal-Cyst-Tumor-Stone"
)

print("Dataset path:", DATASET_PATH)
print("Path exists:", os.path.exists(DATASET_PATH))

if os.path.exists(DATASET_PATH):
    print("Contents:", os.listdir(DATASET_PATH))


Dataset path: artifacts\data_ingestion\CT-KIDNEY-DATASET-Normal-Cyst-Tumor-Stone\CT-KIDNEY-DATASET-Normal-Cyst-Tumor-Stone
Path exists: True
Contents: ['Cyst', 'Normal', 'Stone', 'Tumor']


In [61]:

import os

for class_name in ["Cyst", "Normal", "Stone", "Tumor"]:
    class_path = os.path.join(DATASET_PATH, class_name)
    files = os.listdir(class_path) if os.path.isdir(class_path) else []
    print(f"{class_name}: {len(files)} files")


Cyst: 3709 files
Normal: 5077 files
Stone: 1377 files
Tumor: 2283 files


In [62]:

from pathlib import Path
from dataclasses import replace

config = ConfigurationManager()
training_config = config.get_training_config()

# Create a new configuration with the correct dataset path
training_config = replace(
    training_config,
    training_data=Path(
        "artifacts/data_ingestion/"
        "CT-KIDNEY-DATASET-Normal-Cyst-Tumor-Stone/"
        "CT-KIDNEY-DATASET-Normal-Cyst-Tumor-Stone"
    )
)

print("Training path:", training_config.training_data)
print("Path exists:", training_config.training_data.exists())


[2026-10-12 19:46:07,590: INFO: common: yaml file: config\config.yaml loaded successfully]
[2026-10-12 19:46:07,603: INFO: common: yaml file: params.yaml loaded successfully]
[2026-10-12 19:46:07,610: INFO: common: created directory at: artifacts]
[2026-10-12 19:46:07,614: INFO: common: created directory at: artifacts\training]
Training path: artifacts\data_ingestion\CT-KIDNEY-DATASET-Normal-Cyst-Tumor-Stone\CT-KIDNEY-DATASET-Normal-Cyst-Tumor-Stone
Path exists: True


In [12]:
from pathlib import Path
from dataclasses import replace

try:
    config = ConfigurationManager()
    training_config = config.get_training_config()

    # Use the correct dataset path
    training_config = replace(
        training_config,
        training_data=Path(
            "artifacts/data_ingestion/"
            "CT-KIDNEY-DATASET-Normal-Cyst-Tumor-Stone/"
            "CT-KIDNEY-DATASET-Normal-Cyst-Tumor-Stone"
        )
    )

    training = Training(config=training_config)
    training.get_base_model()
    training.train_valid_generator()
    training.train()
    
except Exception:
    raise


[2026-10-18 10:54:06,815: INFO: common: yaml file: config\config.yaml loaded successfully]
[2026-10-18 10:54:06,827: INFO: common: yaml file: params.yaml loaded successfully]
[2026-10-18 10:54:06,832: INFO: common: created directory at: artifacts]
[2026-10-18 10:54:06,836: INFO: common: created directory at: artifacts\training]
Found 2487 images belonging to 4 classes.
Found 9959 images belonging to 4 classes.
Epoch 1/2
623/623 [==============================] - ETA: 0s - loss: 219.9558 - accuracy: 0.6869
Epoch 1: val_loss improved from inf to 438.77271, saving model to artifacts\training\model.h5
623/623 [==============================] - 2426s 4s/step - loss: 219.9558 - accuracy: 0.6869 - val_loss: 438.7727 - val_accuracy: 0.5010
Epoch 2/2
623/623 [==============================] - ETA: 0s - loss: 109.5788 - accuracy: 0.8073
Epoch 2: val_loss improved from 438.77271 to 382.78503, saving model to artifacts\training\model.h5
623/623 [==============================] - 2584s 4s/step - lo

In [13]:
from pathlib import Path
import tensorflow as tf

model_path = Path("artifacts/training/model.h5")

print("Model exists:", model_path.exists())

if model_path.exists():
    print("Model size:", round(model_path.stat().st_size / (1024 * 1024), 2), "MB")

    model = tf.keras.models.load_model(model_path)
    print("Output shape:", model.output_shape)

Model exists: True
Model size: 56.97 MB
Output shape: (None, 4)


In [14]:
import shutil

shutil.copy2(
    "artifacts/training/model.h5",
    "artifacts/training/model_v1.h5"
)

print("Backup created successfully!")

Backup created successfully!


In [15]:
print("Epochs:", training_config.params_epochs)
print("Base model:", training_config.updated_base_model_path)
print("Training output:", training_config.trained_model_path)

Epochs: 2
Base model: artifacts\prepare_base_model\base_model_updated.h5
Training output: artifacts\training\model.h5


In [16]:
from pathlib import Path
from dataclasses import replace

training_config = replace(
    training_config,
    trained_model_path=Path("artifacts/training/model_v2.h5")
)

print("New output:", training_config.trained_model_path)

New output: artifacts\training\model_v2.h5


In [17]:
training = Training(config=training_config)

training.get_base_model()
training.train_valid_generator()

Found 2487 images belonging to 4 classes.
Found 9959 images belonging to 4 classes.


In [18]:
training.train()

Epoch 1/2
623/623 [==============================] - ETA: 0s - loss: 215.0356 - accuracy: 0.6907
Epoch 1: val_loss improved from inf to 385.36734, saving model to artifacts\training\model_v2.h5
623/623 [==============================] - 2667s 4s/step - loss: 215.0356 - accuracy: 0.6907 - val_loss: 385.3673 - val_accuracy: 0.5790
Epoch 2/2
623/623 [==============================] - ETA: 0s - loss: 112.0081 - accuracy: 0.8056
Epoch 2: val_loss improved from 385.36734 to 308.85254, saving model to artifacts\training\model_v2.h5
623/623 [==============================] - 3111s 5s/step - loss: 112.0081 - accuracy: 0.8056 - val_loss: 308.8525 - val_accuracy: 0.7037


In [ ]:
import os

path = r"artifacts\data_ingestion"
print("Directory exists:", os.path.exists(path))

if os.path.exists(path):
    print(os.listdir(path))

Directory exists: True
['CT-KIDNEY-DATASET-Normal-Cyst-Tumor-Stone', 'data.zip', 'data.zip1naqqm95.part', 'data.zipceadqvv1.part', 'kidneyData.csv']


In [65]:
import inspect

print(inspect.getsource(Training.get_base_model))

    def get_base_model(self):
        self.model = tf.keras.models.load_model(
            self.config.updated_base_model_path
        )



In [66]:

print("Updated model path:", training_config.updated_base_model_path)
print("Model file exists:", training_config.updated_base_model_path.exists())
print("Model output shape:", training.model.output_shape)


Updated model path: artifacts\prepare_base_model\base_model_updated.h5
Model file exists: True
Model output shape: (None, 4)


In [67]:

import tensorflow as tf

model_path = str(training_config.updated_base_model_path)

saved_model = tf.keras.models.load_model(model_path)

print("Saved model output shape:", saved_model.output_shape)


Saved model output shape: (None, 4)


In [68]:

training = Training(config=training_config)
training.get_base_model()

print("Training model output shape:", training.model.output_shape)


Training model output shape: (None, 4)


In [70]:
config = ConfigurationManager()

print("CLASSES from params.yaml:", config.params.CLASSES)

[2026-10-12 19:46:33,189: INFO: common: yaml file: config\config.yaml loaded successfully]
[2026-10-12 19:46:33,198: INFO: common: yaml file: params.yaml loaded successfully]
[2026-10-12 19:46:33,203: INFO: common: created directory at: artifacts]
CLASSES from params.yaml: 4


In [71]:
print("Current model output:", training.model.output_shape)

Current model output: (None, 4)


In [4]:
from pathlib import Path

model_path = Path(
    r"C:\Users\amrut\Downloads\kidney-Disease-Classifiaction-DLProject\artifacts\training\model_v2.h5"
)

print("Model exists:", model_path.exists())

Model exists: True
